# Lesson 03 Lab — PagedAttention and Block Tables

**Puzzle:** How can non-contiguous KV blocks reduce waste without changing attention semantics?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Requests end at unpredictable lengths. Reserving one contiguous maximum-length slab per request strands memory, while moving live cache entries to repair fragmentation is expensive. PagedAttention introduces an indirection that lets logical token positions map to physical blocks.


## 0. Predict before running

1. Compute slab waste for the supplied length distribution.
2. Predict how block size changes waste and metadata.
3. Verify reconstruction after random physical placement.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The notebook runs a deterministic allocator model for a mixed-length batch. It compares maximum-length slabs, exact variable allocations, and fixed-size blocks, then verifies that a shuffled block table reconstructs the same logical sequence.

- Internal fragmentation is bounded by one partial block per sequence.
- External fragmentation is handled by assigning any free physical block.
- Block tables preserve logical order even when physical IDs are shuffled.


## 2. Derive the mechanism

For block size `B`, a request of length `L` owns `ceil(L/B)` blocks and wastes fewer than `B` token slots. The block table maps each logical block number to a physical block. Attention follows that mapping when reading keys and values; physical adjacency is unnecessary. This changes allocation and addressing, not the mathematical attention weights.

### Mechanism at a glance

```mermaid
flowchart LR
  L0["logical block 0"] -->|table| P7["physical block 7"]
  L1["logical block 1"] -->|table| P2["physical block 2"]
  L2["logical block 2"] -->|table| P9["physical block 9"]
  P7 --> A["attention reads logical order"]
  P2 --> A
  P9 --> A
```

### Walk it step by step

1. **Partition logical positions.** Group token positions into equal-size logical blocks.
2. **Allocate from a pool.** Assign each logical block any available physical block.
3. **Follow the table.** Gather physical blocks in logical order inside attention.
4. **Account for the tail.** Only the last block of each request can be partially unused.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 3
LESSON_TITLE = 'PagedAttention and Block Tables'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260815
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | one max-sequence contiguous reservation per request |
| Candidate | fixed-size paged blocks assigned from a shared pool |
| Held constant | request lengths, element footprint, seed, and logical payload |
| Measurements | reserved token slots, waste ratio, block count, and reconstruction equality |
| Evidence | `numerical-model` |

**Experiment:** Simulate slab and block allocation, then reconstruct logical token IDs through a randomized block table.


## 5. Inspect the experiment code

The code treats every token slot as a visible integer, places blocks at non-contiguous physical IDs, and gathers them through the table. This makes address translation inspectable without claiming a CUDA kernel trace.

Do not execute until the code matches the frozen table.


In [2]:
lengths = [33, 81, 127, 130, 255, 401, 700, 997]; max_len = 1024; block_size = 16
slab_reserved = len(lengths) * max_len
paged_reserved = sum(math.ceil(length / block_size) * block_size for length in lengths)
block_counts = [math.ceil(length / block_size) for length in lengths]
physical_ids = list(range(sum(block_counts) + 11)); random.shuffle(physical_ids)
physical = {}; tables = []; cursor = 0
for request_id, (length, blocks) in enumerate(zip(lengths, block_counts)):
    table = physical_ids[cursor:cursor + blocks]; cursor += blocks; tables.append(table)
    payload = list(range(request_id * 10000, request_id * 10000 + length))
    for logical, pid in enumerate(table):
        piece = payload[logical*block_size:(logical+1)*block_size]
        physical[pid] = piece + [-1] * (block_size - len(piece))
reconstructed = [[x for pid in table for x in physical[pid]][:length]
                 for length, table in zip(lengths, tables)]
expected = [list(range(i*10000, i*10000+length)) for i, length in enumerate(lengths)]
metrics = {"request_lengths": lengths, "block_size": block_size,
           "slab_reserved_tokens": slab_reserved, "paged_reserved_tokens": paged_reserved,
           "slab_waste_ratio": (slab_reserved-sum(lengths))/slab_reserved,
           "paged_waste_ratio": (paged_reserved-sum(lengths))/paged_reserved,
           "physical_blocks": sum(block_counts), "block_tables": tables,
           "reconstruction_exact": reconstructed == expected}
analysis = (f"Slabs reserved {slab_reserved:,} positions with {metrics['slab_waste_ratio']:.1%} "
            f"waste; {block_size}-token pages reserved {paged_reserved:,} with "
            f"{metrics['paged_waste_ratio']:.1%} waste. Non-contiguous reconstruction was exact; "
            "this is an allocator model, not a kernel benchmark.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Slab reserved tokens | 8,192 |
| Paged reserved tokens | 2,800 |
| Slab waste | 66.75% |
| Paged waste | 2.71% |
| Physical blocks | 175 |
| Reconstruction exact | yes |


## 7. Explain the result

Slabs reserved 8,192 positions with 66.7% waste; 16-token pages reserved 2,800 with 2.7% waste. Non-contiguous reconstruction was exact; this is an allocator model, not a kernel benchmark.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. A transparent allocator, scheduler, gateway, or policy model executed. It establishes the stated invariant, not native vLLM performance.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 3, "title": 'PagedAttention and Block Tables', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Paged blocks bound per-request tail waste and allow non-contiguous placement; the numerical reconstruction proves the mapping invariant, not native PagedAttention speed.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 3,
  "title": "PagedAttention and Block Tables",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260815
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "request_lengths": [
      33,
      81,
      127,
      130,
      255,
      401,
      700,
      997
    ],
    "block_size": 16,
    "slab_reserved_tokens": 8192,
    "paged_reserved_tokens": 2800,
    "slab_waste_ratio": 0.66748046875,
    "paged_waste_ratio": 0.027142857142857142,
    "physical_blocks": 175,
    "block_tables": [
      [
        94,
        167,
        148
      ],
      [
        13,
        158,
        33,
        102,
        41,
        146
      ],
      [
        178,
        28,
        101,
        58,
        69,
        160,
        107,
        121
      ],
      [
      

## 9. Make the bounded decision

> Paged blocks bound per-request tail waste and allow non-contiguous placement; the numerical reconstruction proves the mapping invariant, not native PagedAttention speed.

**Acceptance/rollback:** Select a block size only after both fragmentation and scheduler/kernel constraints are evaluated on the target distribution.

**Failure analysis:** The allocator model omits copy-on-write, prefix sharing, eviction, block metadata bytes, alignment, and kernel execution. It teaches the invariant but cannot predict native latency.


## 10. Extend the evidence

Collect live request lengths and engine cache metrics, sweep supported block sizes, and add prefix-cache sharing plus eviction events.

The full boundary and references are in [`README.md`](README.md).
